# Carregar e organizar os dados

In [5]:

import pandas as pd
from sklearn.model_selection import train_test_split

# Configurações do usuário
FILE_PATH = "rccarbonation.xlsx"   # caminho do arquivo
SHEET_NAME = "Sheet1"              # aba que deseja carregar
TARGET = "Possan"                          # coluna alvo (y)

# Carregar dataset
df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)

# Separar X e y
X = df.drop(columns=[TARGET])
print(X)

y = df[TARGET]

# Dividir em treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Dataset carregado:", df.shape)
print("Treino:", X_train.shape, "Teste:", X_test.shape)


       CCO₂ (%)  fc (MPa)  RH (%) Type of cement Exposure conditions  \
0          0.08      25.9    35.4         CPII Z                PIA    
1          0.07      21.7    43.8      CPV - ARI                UEA    
2          0.08      30.3    83.7           CPIV                UEA    
3          0.15      22.8    37.6           CPIV                PIA    
4          0.03      26.4    65.2         CPII F                UEA    
...         ...       ...     ...            ...                 ...   
19995      0.19      39.2    43.6         CPII Z                PIA    
19996      0.15      45.0    35.3         CPII F                PIA    
19997      0.28      36.0    42.9           CPIV                PIA    
19998      0.16      34.1    84.9      CPV - ARI                PEA    
19999      0.18      26.3    51.8         CPII E                UEA    

       t (years)  
0              9  
1             92  
2             58  
3             42  
4             96  
...          ...  
19

# Escalar os dados

In [2]:

from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Opção do usuário
scaler_option = "zscore"   # escolha: "zscore" ou "minmax"

if scaler_option == "zscore":
    scaler = StandardScaler()
elif scaler_option == "minmax":
    scaler = MinMaxScaler()
else:
    raise ValueError("Opção inválida: use 'zscore' ou 'minmax'")

# Ajusta o scaler com treino e aplica em treino e teste
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# Treinar e salvar o melhor modelo

In [ ]:


from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import joblib

# Lista de modelos


resultados = []

# Treinar, avaliar
for nome, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)

    mse = mean_squared_error(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    resultados.append((nome, mse, mae, r2))

# Mostrar resultados
resultados_df = pd.DataFrame(resultados, columns=["Modelo", "MSE", "MAE", "R2"])
display(resultados_df.sort_values(by="R2", ascending=False))

# Selecionar melhor modelo
melhor_nome, _, _, _ = resultados_df.sort_values(by="R2", ascending=False).iloc[0]
melhor_modelo = modelos[melhor_nome].fit(X_train_scaled, y_train)
print(f"Melhor modelo: {melhor_nome}")

# Salvar melhor modelo
import joblib
joblib.dump(melhor_modelo, f"{melhor_nome}_best.joblib")
print(f"Melhor modelo salvo: {melhor_nome}_best.joblib")


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000079 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 242
[LightGBM] [Info] Number of data points in the train set: 272, number of used features: 8
[LightGBM] [Info] Start training from score 0.261875
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

,Modelo,MSE,MAE,R2
6,HistGradientBoosting,0.002492,0.037455,0.829591
7,LGBM,0.002537,0.038334,0.826496
5,GradientBoosting,0.002740,0.038062,0.812585
4,RandomForest,0.003289,0.041637,0.775057
3,DecisionTree,0.004378,0.046775,0.700598
0,LinearRegression,0.004922,0.048401,0.663374
1,Ridge,0.004926,0.048116,0.663116
2,Lasso,0.014623,0.083234,-0.000069


Melhor modelo: HistGradientBoosting
Melhor modelo salvo: HistGradientBoosting_best.joblib


# Carregar melhor modelo e prever em todo o dataset

In [4]:

# Carregar melhor modelo salvo
modelo = joblib.load(f"{melhor_nome}_best.joblib")

# Fazer previsões em TODO o dataset (na ordem original)
X_scaled = scaler.fit_transform(X)   # reescalar todo dataset
y_pred_all = modelo.predict(X_scaled)

# Criar dataframe com resultados
df_pred = df.copy()
df_pred[f"Pred_{melhor_nome}"] = y_pred_all

# Salvar em Excel
df_pred.to_excel("resultado_predicoes.xlsx", index=False)

print("Arquivo salvo: resultado_predicoes.xlsx")


Arquivo salvo: resultado_predicoes.xlsx
